# MCP Primitives: Tools, Resources, and Prompts

## Table of contents
- [The three primitives — overview](#the-three-primitives--overview)
- [Tool — model-controlled actions](#tool--model-controlled-actions)
- [Resource — application-controlled data](#resource--application-controlled-data)
- [Prompt — user-controlled templates](#prompt--user-controlled-templates)
- [Judgment framework](#judgment-framework)
- [Setup](#setup)
- [Live demo: Claude calling a Tool](#live-demo-claude-calling-a-tool)
- [Exam trap exercises](#exam-trap-exercises)
- [All three primitives in one domain](#all-three-primitives-in-one-domain)
- [Quick reference](#quick-reference)

## The three primitives — overview

MCP defines three primitive types. Each maps to a different **controller**,
a different **interaction pattern**, and different **side-effect semantics**.

| Primitive | Controller | Side effects? | Selection keyword |
|---|---|---|---|
| **Tool** | Model (Claude decides when to call) | ✅ Yes | write, update, call, execute, send |
| **Resource** | Application (host injects as context) | ❌ No | read, get, list, fetch, retrieve |
| **Prompt** | User (manually selects from UI) | ❌ No | template, workflow, standard process |

> **Exam pattern**: MCP questions give you a scenario and ask which primitive to use.
> The controller and side-effect columns are the fastest path to the correct answer.

## Tool — model-controlled actions

**Controller**: The model (Claude) decides *when* and *how* to call a Tool.

**Side effects**: Yes — Tools can write to databases, call external APIs, send emails,
execute code, or modify any external state.

**API mapping**: MCP Tools use the exact same `tool_use` / `tool_result` mechanism as
raw Claude API tool calls. The MCP layer adds server-side registration and discovery;
the underlying API interaction is identical.

**When to choose Tool**:
- The operation changes something (write, delete, update, send, execute)
- Claude needs to decide when to perform the action based on conversation context
- You want the model to autonomously chain multiple calls

In [ ]:
// MCP Tool primitive — TypeScript type (mirrors @modelcontextprotocol/sdk)
interface McpTool {
  name: string;
  description: string;
  inputSchema: {
    type: 'object';
    properties: Record<string, { type: string; description: string; enum?: string[] }>;
    required: string[];
  };
}

const writeDbTool: McpTool = {
  name: 'write_to_database',
  description:
    'Insert or update a record in the users database. ' +
    'Use this when the user wants to save or update profile information.',
  inputSchema: {
    type: 'object',
    properties: {
      user_id: { type: 'string', description: 'The unique user identifier' },
      field: { type: 'string', description: 'The field to update, e.g. "email"' },
      value: { type: 'string', description: 'The new value for the field' },
    },
    required: ['user_id', 'field', 'value'],
  },
};

const sendEmailTool: McpTool = {
  name: 'send_email',
  description:
    'Send an email to a recipient. ' +
    'Use this when the user wants to send a message or notification.',
  inputSchema: {
    type: 'object',
    properties: {
      to: { type: 'string', description: 'Recipient email address' },
      subject: { type: 'string', description: 'Email subject line' },
      body: { type: 'string', description: 'Email body content' },
    },
    required: ['to', 'subject', 'body'],
  },
};

console.log('Tool examples (model-controlled, have side effects):');
for (const tool of [writeDbTool, sendEmailTool]) {
  console.log(`  name     : ${tool.name}`);
  console.log(`  required : [${tool.inputSchema.required.join(', ')}]`);
  console.log(`  purpose  : "${tool.description.slice(0, 55)}..."`);
  console.log();
}

## Resource — application-controlled data

**Controller**: The application (host) decides *when* to provide a Resource to Claude as context.
Claude does not choose to "call" a Resource — the host injects it.

**Side effects**: None — Resources are read-only. They expose data without modifying any state.

**Addressing**: Resources use URI schemes (`file://`, `db://`, `api://`) and carry a MIME type
so the host knows how to display or handle the content.

**Subscription**: Resources can support subscription — the host notifies Claude when a
resource's content changes (useful for live data feeds).

**When to choose Resource**:
- Providing read-only data as context (files, DB rows, config, documentation)
- The data already exists and won't be changed by this interaction
- The *host*, not Claude, controls when and whether to include the data

In [ ]:
// MCP Resource primitive — TypeScript type (mirrors @modelcontextprotocol/sdk)
interface McpResource {
  uri: string;        // 'file:///data/report.csv', 'db://users/42', 'api://config/v1'
  name: string;
  description: string;
  mimeType: string;   // 'text/csv', 'application/json', 'text/plain'
}

const resources: McpResource[] = [
  {
    uri: 'file:///data/portfolio.csv',
    name: 'portfolio-snapshot',
    description: 'Current portfolio holdings as of last market close',
    mimeType: 'text/csv',
  },
  {
    uri: 'db://users/42',
    name: 'user-profile',
    description: 'Profile data for user ID 42 — injected by host before conversation',
    mimeType: 'application/json',
  },
  {
    uri: 'api://config/current',
    name: 'app-config',
    description: 'Current application configuration — read-only snapshot',
    mimeType: 'application/json',
  },
];

console.log('Resource examples (app-controlled, read-only, no side effects):');
for (const r of resources) {
  console.log(`  name     : ${r.name}  (${r.mimeType})`);
  console.log(`  uri      : ${r.uri}`);
  console.log(`  desc     : ${r.description}`);
  console.log();
}

## Prompt — user-controlled templates

**Controller**: The user — they manually select a Prompt from the application UI.

**Side effects**: None — Prompts are workflow templates that generate structured messages.
They don't execute anything directly.

**Parameters**: Prompts accept arguments to dynamically fill template placeholders,
letting users customize the workflow at trigger time.

**When to choose Prompt**:
- Providing reusable, structured workflows that users trigger manually
- Standard operating procedures (e.g. "Code Review", "Bug Report Analysis")
- Scenarios where a human should consciously choose to start a workflow

> **Key distinction**: Prompts are the only primitive where the *user* is in control.
> Tools = model-controlled. Resources = app-controlled. Prompts = user-controlled.

In [ ]:
// MCP Prompt primitive — TypeScript type (mirrors @modelcontextprotocol/sdk)
interface McpPrompt {
  name: string;
  description: string;
  arguments: Array<{
    name: string;
    description: string;
    required: boolean;
  }>;
}

const prompts: McpPrompt[] = [
  {
    name: 'quarterly-report-analysis',
    description:
      'Standard workflow for analyzing quarterly portfolio performance. ' +
      'User selects this to run a comprehensive Q-over-Q review.',
    arguments: [
      { name: 'quarter', description: 'Quarter to analyze, e.g. "Q1 2026"', required: true },
      { name: 'focus', description: 'Optional focus: "growth" | "risk" | "dividend"', required: false },
    ],
  },
  {
    name: 'code-review',
    description: 'Structured code review workflow — user selects from the review menu.',
    arguments: [
      { name: 'language', description: 'Programming language, e.g. "TypeScript"', required: true },
      { name: 'scope', description: 'Review scope: "security" | "performance" | "style"', required: false },
    ],
  },
];

console.log('Prompt examples (user-controlled, reusable templates, no side effects):');
for (const p of prompts) {
  const req = p.arguments.filter(a => a.required).map(a => a.name);
  const opt = p.arguments.filter(a => !a.required).map(a => a.name);
  console.log(`  name          : ${p.name}`);
  console.log(`  required args : [${req.join(', ')}]`);
  console.log(`  optional args : [${opt.join(', ')}]`);
  console.log();
}

## Judgment framework

Use this decision tree for any primitive-selection question:

```
Does the operation change external state (write / delete / update / send / execute)?
  ├── YES  →  Tool      (model-controlled, has side effects)
  └── NO   →  Does it expose existing data (file / DB row / config / API read)?
                ├── YES  →  Resource   (app-controlled, read-only, URI-addressed)
                └── NO   →  Is it a user-selectable workflow template?
                              └── YES  →  Prompt   (user-controlled, reusable template)
```

### Exam traps (high-frequency)

| Scenario | Wrong answer | Correct answer | Why |
|---|---|---|---|
| Claude reads a CSV file | Tool (sounds like an action) | **Resource** | Read-only, no side effects |
| Claude writes results to a DB | Resource (sounds like data) | **Tool** | Has side effects (write) |
| Analysts trigger "Quarterly Report" from UI | Tool (Claude does the work) | **Prompt** | User-selected workflow template |

> **Memory hook**: Only the *controller* changes — model, app, user. Match the controller to the primitive.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });

console.log('Client ready. Model:', MODEL_NAME);

In [ ]:
// Live demo: Claude calling an MCP-style Tool
// An MCP Tool converted to Claude API format looks identical to a raw tool_use schema.

const analyzePortfolioTool: Anthropic.Tool = {
  name: 'analyze_portfolio',
  description:
    'Calculate portfolio performance metrics for a given time period. ' +
    'Use this when the user asks to analyze investment performance.',
  input_schema: {
    type: 'object',
    properties: {
      period: {
        type: 'string',
        description: 'Analysis period, e.g. "Q1 2026" or "2025"',
      },
      metric: {
        type: 'string',
        enum: ['return', 'risk', 'sharpe_ratio'],
        description: 'Performance metric to calculate',
      },
    },
    required: ['period', 'metric'],
  },
};

function analyzePortfolio(period: string, metric: string): object {
  const data: Record<string, Record<string, number>> = {
    'Q1 2026': { return: 4.2, risk: 12.1, sharpe_ratio: 0.85 },
    '2025': { return: 18.5, risk: 15.3, sharpe_ratio: 1.21 },
  };
  const values = data[period] ?? { return: 7.3, risk: 13.5, sharpe_ratio: 0.94 };
  return { period, metric, value: values[metric] ?? 0, unit: metric === 'return' ? '%' : 'ratio' };
}

const messages: Anthropic.MessageParam[] = [
  { role: 'user', content: 'What was the portfolio Sharpe ratio for Q1 2026?' },
];

while (true) {
  const response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 512,
    tools: [analyzePortfolioTool],
    messages,
  });

  console.log('stop_reason:', response.stop_reason);

  if (response.stop_reason === 'end_turn') {
    const text = response.content.find(b => b.type === 'text');
    console.log('\nFinal answer:', text?.type === 'text' ? text.text : '(no text)');
    break;
  }

  messages.push({ role: 'assistant', content: response.content });

  const toolResults: Anthropic.ToolResultBlockParam[] = [];
  for (const block of response.content) {
    if (block.type !== 'tool_use') continue;
    const input = block.input as { period: string; metric: string };
    console.log(`  tool_use: ${block.name}(period="${input.period}", metric="${input.metric}")`);
    const result = analyzePortfolio(input.period, input.metric);
    console.log('  tool_result:', result);
    toolResults.push({
      type: 'tool_result',
      tool_use_id: block.id,
      content: JSON.stringify(result),
    });
  }
  messages.push({ role: 'user', content: toolResults });
}

## Exam trap exercises

Each scenario below maps to exactly one primitive: **Tool**, **Resource**, or **Prompt**.

Try to classify each one before reading the answer. Pay special attention to
the **Common trap** column — these are the distractors the exam uses most often.

In [ ]:
type Primitive = 'Tool' | 'Resource' | 'Prompt';

interface ExamScenario {
  scenario: string;
  commonTrap?: string;
  answer: Primitive;
  reason: string;
}

const scenarios: ExamScenario[] = [
  {
    scenario: 'Claude must read the contents of a CSV file.',
    commonTrap: 'Tool — sounds like an action Claude takes',
    answer: 'Resource',
    reason: 'Read-only data, no side effects, app-controlled URI-addressed source.',
  },
  {
    scenario: 'Claude must write analysis results to a database.',
    commonTrap: 'Resource — database sounds like a "data source"',
    answer: 'Tool',
    reason: 'Write operation has side effects; model decides when to call it.',
  },
  {
    scenario: 'Analysts select "Quarterly Report Analysis" from the application UI.',
    commonTrap: 'Tool — Claude performs the analysis work',
    answer: 'Prompt',
    reason: 'User-selected reusable workflow template = Prompt by definition.',
  },
  {
    scenario: 'Claude sends a notification email after processing completes.',
    answer: 'Tool',
    reason: 'Sending email is a write/execute action with side effects.',
  },
  {
    scenario: 'The host provides Claude with the current app config as context.',
    answer: 'Resource',
    reason: 'App-controlled, read-only injection of existing data — not a model action.',
  },
  {
    scenario: 'A support agent triggers a "Bug Report Intake" template from a menu.',
    commonTrap: 'Tool — sounds operational',
    answer: 'Prompt',
    reason: 'Manually triggered from UI, reusable workflow template = Prompt.',
  },
];

console.log('=== Primitive Classification Exercises ===\n');
for (const [i, s] of scenarios.entries()) {
  console.log(`Q${i + 1}: ${s.scenario}`);
  if (s.commonTrap) console.log(`  Common trap : ${s.commonTrap}`);
  console.log(`  Answer      : ${s.answer}`);
  console.log(`  Reason      : ${s.reason}`);
  console.log();
}

In [ ]:
// Same CRM domain — all three primitives coexisting in one MCP server
interface PrimitiveCard {
  primitive: 'Tool' | 'Resource' | 'Prompt';
  name: string;
  controller: string;
  sideEffects: boolean;
  scenario: string;
}

const crmPrimitives: PrimitiveCard[] = [
  {
    primitive: 'Tool',
    name: 'update_crm_record',
    controller: 'Model (Claude)',
    sideEffects: true,
    scenario: 'Claude updates a CRM contact when the user provides new information.',
  },
  {
    primitive: 'Resource',
    name: 'crm://contacts/42',
    controller: 'Application (host)',
    sideEffects: false,
    scenario: 'Host injects contact data as read-only context before the conversation starts.',
  },
  {
    primitive: 'Prompt',
    name: 'crm-onboarding-flow',
    controller: 'User (UI selection)',
    sideEffects: false,
    scenario: 'Sales rep selects "New Customer Onboarding" workflow template from a menu.',
  },
];

const hdr = `${'Primitive'.padEnd(10)}| ${'Controller'.padEnd(22)}| ${'Side Effects'.padEnd(13)}| Name`;
console.log('=== Same CRM Domain — Three Primitives ===\n');
console.log(hdr);
console.log('-'.repeat(hdr.length));
for (const p of crmPrimitives) {
  console.log(
    `${p.primitive.padEnd(10)}| ${p.controller.padEnd(22)}| ${(p.sideEffects ? 'Yes' : 'No').padEnd(13)}| ${p.name}`,
  );
}
console.log('\nScenarios:');
for (const p of crmPrimitives) {
  console.log(`  [${p.primitive}] ${p.scenario}`);
}
console.log('\nAll three primitives can coexist in the same MCP server — they serve different purposes.');

## Quick Reference

### Three-primitive decision (exam high-frequency)

```
Operation has side effects (write / execute / send / call)?   →  Tool      (model-controlled)
Read-only data (file / DB row / config / API snapshot)?       →  Resource  (app-controlled)
User-selected workflow template?                              →  Prompt    (user-controlled)
```

### Controller cheat sheet

| Controller | Primitive | How it arrives at Claude |
|---|---|---|
| **Model** (Claude decides) | Tool | Claude outputs `tool_use` block |
| **Application** (host injects) | Resource | Host includes as context before/during conversation |
| **User** (manually triggers) | Prompt | User selects from UI menu |

### Classic exam traps to memorize

1. **"Read a CSV"** → Resource (not Tool — no side effects)
2. **"Write to DB"** → Tool (not Resource — has side effects)
3. **"Quarterly report workflow from UI"** → Prompt (not Tool — user-controlled template)

### Next notebooks

- `02_mcp_server_basics.ipynb` — build a working MCP server with all three primitives
- `03_tool_schema_design.ipynb` — spot and fix broken tool schemas (high-frequency exam pattern)
- `04_mcp_client_with_claude.ipynb` — end-to-end: Claude + MCP client + error handling